# Practical Exam: Automating Customer Support with OpenAI API

You work as an AI Engineer at ChatSolveAI, a company that provides automated customer support solutions. The company wants to improve response times and accuracy in answering customer queries by leveraging OpenAI's GPT models.

Your task is to build a chatbot that classifies customer queries, retrieves relevant responses, and logs interactions in a structured way. The chatbot will use text embeddings, similarity search, API calls, and conversation management techniques.


**Please note:** 

1. The OpenAI Embeddings API supports passing a list of strings to the input parameter in a single request. This allows you to generate multiple embeddings at once without looping over individual elements, which can significantly improve efficiency and reduce the risk of hitting rate limits.

2. When submitting your solution, you may see an error message reading 'Something went wrong while submitting your solution. Please try again.' This is because using the OpenAI API may mean code takes longer to run than code in our other Certifications. Please ignore this message if your code is taking a few minutes to run. However, if your code makes too many API requests, the API will time out. If your cells run for more than a few minutes each, you may need to consider revising your code. 

In [1]:
# Run this cell before running your solution

# Import necessary modules
import os
from openai import OpenAI

# Define the model to use
model = "gpt-4o-mini"

# Define the client
client = OpenAI()

# Task 1

ChatSolveAI has provided a knowledge base (`knowledge_base.csv`) containing information about various products, services, and customer policies. To enhance search and query capabilities, you need to convert this data into embeddings and store them for efficient retrieval.

- Load the dataset (`knowledge_base.csv`).
- Generate text embeddings using OpenAI's embedding model (`text-embedding-3-small`). Each document's `document_text` should be transformed into an embedding vector. Do not apply any text transformations such as lowercasing, stripping or normalization before embedding.
- Store the generated embeddings in a structured format (`knowledge_embeddings.json`) with the following format available below.
- Store the embedded data and associated metadata for retrieval.  

### Format to store generated embeddings:
```json
[
    {
       "document_id": 1,
       "document_text": "Example document text.",
       "embedding_vector": [0.123, 0.456, ...],
       "metadata": "Additional document info"
    }
]
```

### Data description: 

| Column Name       | Criteria                                                |
|-------------------|---------------------------------------------------------|
| document_id       | Integer. Unique identifier for each document. No missing values. |
| document_text     | String. Text content of the knowledge base. Preprocessed and embedded. |
| embedding_vector  | List. Embedding representation of the `document_text`. |
| metadata          | String. Metadata for additional information. |


In [2]:
# Write your answer to Task 1 here
import csv
import json
from openai import OpenAI

client = OpenAI()

# Load the knowledge base CSV
knowledge_base = []
with open("knowledge_base.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        knowledge_base.append({
            "document_id": int(row["document_id"]),
            "document_text": row["document_text"],
            "metadata": row["metadata"]
        })

# Extract all document texts (no transformations applied)
document_texts = [doc["document_text"] for doc in knowledge_base]

# Generate embeddings in a single batch request
embedding_response = client.embeddings.create(
    input=document_texts,
    model="text-embedding-3-small"
)

# Build the structured output with embeddings
knowledge_embeddings = []
for i, doc in enumerate(knowledge_base):
    knowledge_embeddings.append({
        "document_id": doc["document_id"],
        "document_text": doc["document_text"],
        "embedding_vector": embedding_response.data[i].embedding,
        "metadata": doc["metadata"]
    })

# Save to knowledge_embeddings.json
with open("knowledge_embeddings.json", "w", encoding="utf-8") as f:
    json.dump(knowledge_embeddings, f, indent=2)

print(f"Saved {len(knowledge_embeddings)} embeddings to knowledge_embeddings.json")

# Task 2

ChatSolveAI receives customer queries that need to be classified and matched with appropriate responses. Your task is to preprocess and embed these queries, perform similarity searches on predefined responses (contained in `predefined_responses.json`), and retrieve the most relevant responses.

- Load the dataset (`processed_queries.csv`).
- Retrieve responses by using cosine similarity to perform a similarity search against predefined responses in `predefined_responses.json`.
- Structure API requests properly and implement error handling, including retry mechanisms to handle rate limits.
- Format model responses as JSON to maintain consistency in output.
- Compute confidence scores for retrieved responses, scaled to 0-1.
- Store the structured responses in a JSON file (`query_responses.json`), suitable for integration with other applications. Your JSON file should be structured as follows:

| Column Name       | Criteria                                                   |
|-------------------|------------------------------------------------------------|
| query_id         | Integer. Unique identifier for each query. No missing values. |
| query_text       | String. Preprocessed query text. |
| top_responses    | List. Top 3 most relevant response strings retrieved. |
| confidence_scores | List. Model-based confidence score for the top 3 responses. |

In [3]:
# Write your answer to Task 2 here
import csv
import json
import math
import time
from openai import OpenAI

client = OpenAI()

# Load processed queries
queries = []
with open("processed_queries.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        queries.append({
            "query_id": int(row["query_id"]),
            "query_text": row["query_text"]
        })

# Load predefined responses
with open("predefined_responses.json", encoding="utf-8") as f:
    predefined_responses = json.load(f)

response_texts = list(predefined_responses.values())

def embed_with_retry(texts, model="text-embedding-3-small", max_retries=3):
    """Embed a list of texts with retry logic for rate limits."""
    for attempt in range(max_retries):
        try:
            result = client.embeddings.create(input=texts, model=model)
            return [item.embedding for item in result.data]
        except Exception as e:
            if "rate_limit" in str(e).lower() and attempt < max_retries - 1:
                wait_time = 2 ** attempt  # exponential backoff
                print(f"Rate limit hit, retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                raise

# Embed all predefined responses in one batch
response_embeddings = embed_with_retry(response_texts)

# Embed all query texts in one batch
query_texts = [q["query_text"] for q in queries]
query_embeddings = embed_with_retry(query_texts)

def cosine_similarity(vec_a, vec_b):
    """Compute cosine similarity between two vectors."""
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    mag_a = math.sqrt(sum(a * a for a in vec_a))
    mag_b = math.sqrt(sum(b * b for b in vec_b))
    if mag_a == 0 or mag_b == 0:
        return 0.0
    return dot / (mag_a * mag_b)

# For each query, find top 3 most similar predefined responses
query_responses = []
for i, query in enumerate(queries):
    q_emb = query_embeddings[i]
    similarities = []
    for j, r_emb in enumerate(response_embeddings):
        sim = cosine_similarity(q_emb, r_emb)
        similarities.append((response_texts[j], sim))
    # Sort by similarity descending and take top 3
    similarities.sort(key=lambda x: x[1], reverse=True)
    top_3 = similarities[:3]
    query_responses.append({
        "query_id": query["query_id"],
        "query_text": query["query_text"],
        "top_responses": [item[0] for item in top_3],
        "confidence_scores": [round(max(0.0, min(1.0, item[1])), 4) for item in top_3]
    })

# Save to query_responses.json
with open("query_responses.json", "w", encoding="utf-8") as f:
    json.dump(query_responses, f, indent=2)

print(f"Saved {len(query_responses)} query responses to query_responses.json")

# Task 3

To provide seamless customer service, ChatSolveAI wants to develop a chatbot that can respond to customer queries efficiently by searching for relevant responses and generating new ones when necessary.

- Develop a chatbot that:
    - Accepts customer queries via text input.
    - Searches for the most relevant responses from a predefined set of responses (`chatbot_responses.json`).
    - Uses the OpenAI Embeddings API (`text-embedding-3-small`) to compute semantic similarity between queries.
    - If no relevant response is found from the predefined set, generates a new response using GPT-3.5-turbo.
- Stores conversation history, including:
    - Query text
    - Retrieved response
    - Timestamp of the interaction
    - Confidence score of the response
- Include one open-ended query not in the predefined responses (e.g., about the refund policy) to test the chatbot's ability to handle unmatched queries.
- Include one paraphrased query about support hours (e.g., “When can I talk to someone from support?”) to test semantic similarity matching.
- Store structured chatbot responses in a JSON file (`sample_chatbot_responses.json`). Make sure they follow this format:
```json
[
    {
        "query_text": "How do I reset my password?",
        "retrieved_response": "You can reset your password by clicking 'Forgot Password' on the login page.",
        "timestamp": "2025-04-02T14:30:00Z",
        "confidence_score": 0.92
    },
    {
        "query_text": "What are your business hours?",
        "retrieved_response": "Our support team is available from 9 AM to 5 PM, Monday to Friday.",
        "timestamp": "2025-04-02T14:35:00Z",
        "confidence_score": 0.87
    }
]
```

In [4]:
# Write your answer to Task 3 here
import json
import math
from datetime import datetime, timezone
from openai import OpenAI

client = OpenAI()

# Load predefined chatbot responses
with open("chatbot_responses.json", encoding="utf-8") as f:
    chatbot_responses = json.load(f)

predefined_queries = [item["query_text"] for item in chatbot_responses]
predefined_answers = [item["retrieved_response"] for item in chatbot_responses]

# Embed all predefined queries in one batch
pred_emb = client.embeddings.create(
    input=predefined_queries,
    model="text-embedding-3-small"
)
predefined_embeddings = [item.embedding for item in pred_emb.data]

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    dot = sum(x * y for x, y in zip(a, b))
    ma = math.sqrt(sum(x * x for x in a))
    mb = math.sqrt(sum(x * x for x in b))
    if ma == 0 or mb == 0:
        return 0.0
    return dot / (ma * mb)

# Threshold for considering a predefined response as relevant
SIMILARITY_THRESHOLD = 0.75

def chatbot_respond(user_query):
    """Respond to a user query using predefined responses or GPT generation."""
    # Embed the user query
    qe = client.embeddings.create(
        input=[user_query],
        model="text-embedding-3-small"
    )
    q_emb = qe.data[0].embedding

    # Find the most similar predefined query
    best_sim, best_idx = -1, -1
    for j, p_emb in enumerate(predefined_embeddings):
        sim = cosine_similarity(q_emb, p_emb)
        if sim > best_sim:
            best_sim = sim
            best_idx = j

    ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

    if best_sim >= SIMILARITY_THRESHOLD:
        # Use predefined response
        response = predefined_answers[best_idx]
        confidence = round(best_sim, 2)
    else:
        # Generate a new response with GPT-3.5-turbo
        chat_resp = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful customer support assistant."},
                {"role": "user", "content": user_query}
            ],
            temperature=0.7,
            max_tokens=200
        )
        response = chat_resp.choices[0].message.content.strip()
        confidence = round(best_sim, 2)

    return {
        "query_text": user_query,
        "retrieved_response": response,
        "timestamp": ts,
        "confidence_score": confidence
    }

# Test queries:
# 1. Paraphrased query about support hours -> should match via semantic similarity
# 2. Open-ended query about refund policy for international orders -> no close match, triggers GPT
test_queries = [
    "When can I talk to someone from support?",
    "Can you explain your refund policy for international orders?"
]

conversation_history = []
for query in test_queries:
    result = chatbot_respond(query)
    conversation_history.append(result)
    print(f"Query: {result['query_text']}")
    print(f"Response: {result['retrieved_response']}")
    print(f"Confidence: {result['confidence_score']}")
    print(f"Timestamp: {result['timestamp']}")
    print("-" * 60)

# Save conversation history to sample_chatbot_responses.json
with open("sample_chatbot_responses.json", "w", encoding="utf-8") as f:
    json.dump(conversation_history, f, indent=2)

print(f"Saved {len(conversation_history)} responses to sample_chatbot_responses.json")